# 第17章　セグメンテーション結果の定量化と後処理**『医療診断支援AIの社会実装（社会実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-social

## 連結成分と後処理を、コードで

In [ ]:
import numpy as npfrom scipy import ndimagedef quantify(mask, spacing):               # spacing=(sz, sy, sx) [mm]    voxel_vol = np.prod(spacing)           # 1ボクセルの体積[mm^3]    labels, n = ndimage.label(mask)        # 連結成分に分割    results = []    for i in range(1, n + 1):        blob = labels == i        vox = blob.sum()        vol = vox * voxel_vol        if vol < MIN_LESION_VOL:           # 臨床的下限より小さい塊は除外            continue        # 体積から求めた等価球径。臨床でいう最大径（軸位断の最長径）とは別物なので、        # キー名で区別する。同じ "diameter_mm" に入れると、下流で必ず取り違える。        results.append({"volume_mm3": vol,                        "equivalent_sphere_diameter_mm":                            2 * (3 * vol / (4 * np.pi)) ** (1/3)})    return results

## マスクの先の情報 ― ラジオミクス（radiomics）

In [ ]:
import SimpleITK as sitkfrom radiomics import featureextractorextractor = featureextractor.RadiomicsFeatureExtractor()extractor.settings["binWidth"] = 25          # 濃度の量子化幅（再現性に直結）feats = extractor.execute(sitk.ReadImage(ct_path),                          sitk.ReadImage(mask_path))   # 数百特徴を一括抽出